<a href="https://colab.research.google.com/github/DilanSenanayake/CDAZZDEV-MLE-DilanSenanayake/blob/master/task2_genai/finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 2 - Domain Fine-Tuning (QLoRA)

**Use case:** Financial risk-disclosure / compliance Q&A  
**Student:** `microsoft/Phi-3-mini-4k-instruct`  
**Teacher:** Groq `openai/gpt-oss-20b`

## Before you start
1. **Runtime -> Change runtime type -> T4 GPU -> Save**
2. Run cells top-to-bottom
3. After the **Install** cell you **must** restart the runtime, then run the **After restart** cell


In [1]:
# Clone or update repo (Colab)
from pathlib import Path

if not Path("CDAZZDEV-MLE-DilanSenanayake").exists():
    !git clone https://github.com/DilanSenanayake/CDAZZDEV-MLE-DilanSenanayake.git
else:
    !git -C CDAZZDEV-MLE-DilanSenanayake pull

%cd CDAZZDEV-MLE-DilanSenanayake/task2_genai
!pwd
!ls data eval finetune.ipynb


Cloning into 'CDAZZDEV-MLE-DilanSenanayake'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 93 (delta 18), reused 89 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 199.65 KiB | 1.10 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/content/CDAZZDEV-MLE-DilanSenanayake/task2_genai
/content/CDAZZDEV-MLE-DilanSenanayake/task2_genai
finetune.ipynb

data:
split_meta.json  test.jsonl  train.jsonl  val.jsonl

eval:
COMPARISON_TABLE.md  generations.json  metrics.json  qualitative_analysis.md


In [2]:
# GPU check — QLoRA will NOT work on CPU runtime
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    raise RuntimeError("Enable GPU: Runtime -> Change runtime type -> T4 GPU -> Save, then re-run from top")


CUDA available: True
GPU: Tesla T4
CUDA version: 12.8


## Install dependencies

Run the cell below, then **Runtime -> Restart session** (required).


In [3]:
# Pin versions tested on Colab T4; upgrade bitsandbytes for current CUDA wheels
import sys
!{sys.executable} -m pip install -q -U pip
!{sys.executable} -m pip install -q \
    "transformers==4.46.3" "datasets==3.1.0" "peft==0.13.2" "trl==0.11.4" \
    "accelerate==1.1.1" "evaluate" "rouge-score" "huggingface_hub"
# Install latest bitsandbytes (Colab Python 3.13 needs recent CUDA wheel)
!{sys.executable} -m pip install -q -U bitsandbytes
print("Done. NOW: Runtime -> Restart session, then run the NEXT cell only.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.9.0 which is incompatible.
Done. NOW: Runtime -> Restart session, then run the NEXT cell only.


## After restart (run this cell first)

Do **not** skip this after restarting.


In [1]:
# Post-restart setup
import os, sys, torch
from pathlib import Path

REPO = "/content/CDAZZDEV-MLE-DilanSenanayake/task2_genai"
if not Path(REPO).exists():
    REPO = str(Path.cwd() / "CDAZZDEV-MLE-DilanSenanayake" / "task2_genai")
os.chdir(REPO)
print("cwd:", os.getcwd())

assert torch.cuda.is_available(), "GPU not available — enable T4 GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))

import bitsandbytes as bnb
print("bitsandbytes:", bnb.__version__)

import trl, transformers, peft
print("trl:", trl.__version__, "| transformers:", transformers.__version__, "| peft:", peft.__version__)


cwd: /content/CDAZZDEV-MLE-DilanSenanayake/task2_genai
GPU: Tesla T4
bitsandbytes: 0.50.2
trl: 0.11.4 | transformers: 4.46.3 | peft: 0.13.2


## Hyperparameter justification

| Parameter | Value | Reason |
|-----------|-------|--------|
| LoRA r | 16 | Capacity for format learning without overfitting ~100 examples |
| LoRA alpha | 32 | alpha=2r standard PEFT scaling |
| Target modules | attn + MLP | Instruction-following on Phi-3 |
| Learning rate | 2e-4 | Standard QLoRA range |
| LR scheduler | cosine | Reduces late-epoch overfit |
| Epochs | 3 | Enough for scaffold learning on small set |
| Batch size | 1 | T4 VRAM with 4-bit + seq 1024 |
| Grad accumulation | 8 | Effective batch size 8 |
| Max seq length | 1024 | Fits compliance answers |
| Quantization | 4-bit NF4 | Rubric-required QLoRA |
| Warmup ratio | 0.03 | Stabilizes early steps |


In [2]:
# Dataset split check
import json
from pathlib import Path
meta = json.loads(Path('data/split_meta.json').read_text())
print('train/val/test/total:', meta['train'], meta['val'], meta['test'], meta['total'])
print('topics:', len(meta['diversity']['topic_counts']))


train/val/test/total: 96 12 12 120
topics: 20


## QLoRA training (Colab T4 GPU)

In [3]:
# QLoRA fine-tune Phi-3-mini — Colab T4
import torch
from datasets import load_dataset
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig

MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
assert torch.cuda.is_available()

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
if getattr(config, "rope_scaling", None):
    rs = dict(config.rope_scaling)
    if "type" not in rs and "rope_type" in rs:
        rs["type"] = rs["rope_type"]
    config.rope_scaling = rs

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=compute_dtype,
    attn_implementation="eager",
)
print("Model loaded on GPU")

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
model.print_trainable_parameters()

train_ds = load_dataset("json", data_files="data/train.jsonl", split="train")
val_ds = load_dataset("json", data_files="data/val.jsonl", split="train")

def to_text(example):
    parts = [f"<|{m['role']}|>\n{m['content']}<|end|>\n" for m in example["messages"]]
    parts.append("<|assistant|>\n")
    return {"text": "".join(parts)}

train_ds = train_ds.map(to_text)
val_ds = val_ds.map(to_text)

# trl 0.11.4: SFTConfig + tokenizer= (NOT processing_class)
_common = dict(
    output_dir="checkpoints/phi3-compliance-qlora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    save_strategy="epoch",
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    report_to=["none"],
    max_grad_norm=0.3,
    max_seq_length=1024,
    dataset_text_field="text",
)
try:
    sft_args = SFTConfig(**_common, eval_strategy="epoch")
except TypeError:
    sft_args = SFTConfig(**_common, evaluation_strategy="epoch")

trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
)
print("SFTTrainer ready (trl", __import__("trl").__version__, ")")

train_result = trainer.train()
metrics_epoch = trainer.evaluate()
print("Train:", train_result)
print("Eval:", metrics_epoch)

merged = model.merge_and_unload()
merged.save_pretrained("merged_model")
tokenizer.save_pretrained("merged_model")
print("Saved merged_model/")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Epoch,Training Loss,Validation Loss
1,2.400500,1.596191
2,0.858800,0.385992
3,0.313700,0.296874


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Train: TrainOutput(global_step=36, training_loss=1.2440160372191005, metrics={'train_runtime': 1065.7946, 'train_samples_per_second': 0.27, 'train_steps_per_second': 0.034, 'total_flos': 1738076658186240.0, 'train_loss': 1.2440160372191005, 'epoch': 3.0})
Eval: {'eval_loss': 0.2968737483024597, 'eval_runtime': 13.4743, 'eval_samples_per_second': 0.891, 'eval_steps_per_second': 0.891, 'epoch': 3.0}


/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/bnb.py:336: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Saved merged_model/


## Push to Hugging Face (optional)

Set HF token in Colab Secrets as `HF_TOKEN`, then run:


In [4]:
from huggingface_hub import login
from google.colab import userdata
login(userdata.get('HF_TOKEN'))
merged.push_to_hub('DilanSenanayake/phi3-compliance-qlora-merged')
tokenizer.push_to_hub('DilanSenanayake/phi3-compliance-qlora-merged')
print('Pushed to Hugging Face')

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...3r7ecjm/model.safetensors:   1%|          | 15.5MB / 2.66GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...po97lkn2u/tokenizer.model: 100%|##########|  500kB /  500kB            

Pushed to Hugging Face


## Evaluation (Task 2C)

In [5]:
import json
from pathlib import Path
m = json.loads(Path('eval/metrics.json').read_text())
print('ROUGE-L:', m['rougeL'])
print('Keyword F1:', m['keyword_compliance_f1'])
print('Hallucination %:', m['hallucination_rate_pct'])

ROUGE-L: {'base_mean': 0.05527955690656031, 'finetuned_mean': 0.5024618806094847}
Keyword F1: {'base_mean': 0.0, 'finetuned_mean': 1.0}
Hallucination %: 0.0


## Qualitative analysis

Fine-tuning improved structured compliance answers (principle, must/must-not, hallucination guard, caveat) versus generic base responses. Remaining gaps: industry-specific nuance and multi-part questions — more diverse teacher data and preference tuning would help.
